# 1 · Corpus, análisis exploratorio y preprocesado

Práctica final · **Asistente de agenda en lenguaje natural** · Procesamiento del Lenguaje Natural

Poned en la misma carpeta este notebook, `utilidades.py`, `evaluar.py`, `train.jsonl` y `agenda.jsonl`. Las celdas marcadas con `# TODO` son las que tenéis que completar.

In [ ]:
from utilidades import cargar, guardar, anotar, comprobar, INTENCIONES, TIPOS
from collections import Counter
import pandas as pd

train = cargar("train.jsonl")
agenda = cargar("agenda.jsonl")
print(len(train), "frases ·", len(agenda), "entradas en la agenda")

## 1.1 Ampliación del corpus
Añadid al menos 100 frases propias con las convenciones de etiquetado del enunciado. Usad ids propios (`g001`, `g002`…) y marcad con `dificil=True` las que tengan faltas, abreviaturas, negaciones, etc.

In [ ]:
mis_frases = [
    anotar("g001", "Mañana a las 6 tengo pádel en el polideportivo", "crear_evento",
           [("Mañana", "FECHA", "2026-02-10"), ("a las 6", "HORA", "18:00"), ("polideportivo", "LUGAR")]),
    # TODO: al menos 100 frases más, variadas y de todas las intenciones
]
corpus = train + mis_frases
comprobar(corpus)
print(len(corpus), "frases en total")

## 1.2 Estadística del corpus (tema 1)
Frases y palabras por intención, vocabulario, palabras más frecuentes (con y sin stopwords), ley de Zipf, concordancias de expresiones temporales.

In [ ]:
df = pd.DataFrame(corpus)
df["n_palabras"] = df.texto.str.split().str.len()
print(df.intencion.value_counts())
print(df.groupby("intencion").n_palabras.mean().round(1))
# TODO: vocabulario, palabras más frecuentes por intención, Zipf, concordancias

## 1.3 Recursos lingüísticos (tema 2)
Tokenización, POS, lematización y NER con spaCy (`es_core_news_md`). Comparad las entidades de spaCy con vuestras etiquetas.

In [ ]:
# !python -m spacy download es_core_news_md
# import spacy
# nlp = spacy.load("es_core_news_md")
# TODO

## 1.4 Preprocesado (tema 3)
Una función de normalización configurable. Cuando funcione, **copiadla a `utilidades.py`** para usarla en los demás notebooks (se necesita en el experimento E2).

In [ ]:
def normalizar(texto, minusculas=True, quitar_tildes=False, abreviaturas=False, quitar_stopwords=False, raiz=False):
    t = texto.lower() if minusculas else texto
    # TODO: tildes, abreviaturas de mensajería («mñn», «q»), números escritos con letras, stopwords, stemming/lematización
    return t

print(normalizar("q tengo mñn??", abreviaturas=True))

## 1.5 Entrenamiento y validación
Se separa una sola vez y se guarda: **todos** los notebooks usan estas mismas particiones. La validación no se usa nunca para entrenar.

In [ ]:
from sklearn.model_selection import train_test_split
tr, val = train_test_split(corpus, test_size=0.2, random_state=0, stratify=[x["intencion"] for x in corpus])
guardar(tr, "mi_train.jsonl")
guardar(val, "mi_validacion.jsonl")
print(len(tr), "entrenamiento ·", len(val), "validación ·", sum(x["dificil"] for x in val), "difíciles en validación")